In [1]:
import pandas as pd
import glob

# Read all files matching the pattern PRISMAL_REPROCESS*.tsv
file_pattern = "PRISMAL_REPROCESS*.tsv"
files = glob.glob(file_pattern)

# Merge all files into a single DataFrame
dfs = [pd.read_csv(file, sep='\t') for file in files]
merged_df = pd.concat(dfs, ignore_index=True)

# Display the merged DataFrame
merged_df.head()

,rowid,SpectrumFile,Scan,Annotation,Charge,Score,ProtsAll,AnnotationOther,ChargeOther,ScoreOther,ProtsAllOther,MinNTermAdd,minNTermSubtract,MinCTermAdd,minCTermSubtract,internalFilename,id
0,1,xkan/July_mgfs/Normal_1_2_3.mgf,5404,GLESAVIYGSLPPGTK,2.0,80.160301,"(sp|Q8IYB8|SUV3_HUMAN,0,0,0,0);(tr|B1AR60|B1AR...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,input_spectra-00000.mgf,0
1,2,xkan/July_mgfs/Normal_1_2_3.mgf,4132,QVQHILASASPSGR,2.0,74.987846,"(sp|O94979-10|SC31A_HUMAN,0,0,0,0);(sp|O94979-...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,input_spectra-00000.mgf,1
2,3,xkan/July_mgfs/Normal_1_2_3.mgf,17693,ETEEIVSASN+0.984SSR,2.0,66.672752,"(sp|P01266-2|THYG_HUMAN,0,0,0,0);(sp|P01266|TH...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,input_spectra-00000.mgf,2
3,4,xkan/July_mgfs/Normal_1_2_3.mgf,2672,WTELAGC+57.021TADFR,2.0,59.721668,"(sp|Q12907|LMAN2_HUMAN,0,0,0,0);(tr|A0A8Q3WK65...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,input_spectra-00000.mgf,3
4,5,xkan/July_mgfs/Normal_1_2_3.mgf,15307,SGPVMGGGLPPPPIK,2.0,55.519581,"(sp|Q9BYJ9|YTHD1_HUMAN,0,0,0,0)",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,input_spectra-00000.mgf,4


now get a peptide level analysis, with each row as peptide after demodification at first column, then the count_DB, with the number the peptide demod matches the peptide demod in df 'Annotation', and every time it matched put the 'Scan' in new clumn Scans_DB, then count_PA where it matched AnnotationOther, put the Scans_PA from Scan. Over all do this by createing a set of peptide demode from both Annotation and AnnotationOther, then go thourgh the df and add the counts and scan numbers

In [6]:
import re
from tqdm import tqdm

# Function to remove modifications from peptides
def demodify_peptide(peptide):
    return re.sub(r'[\+\-]\d+(\.\d+)?', '', peptide) if isinstance(peptide, str) else None


# Initialize results
results = []

# Add demodified annotations as new columns to the merged DataFrame for quicker search
merged_df['DemodifiedAnnotation'] = merged_df['Annotation'].dropna().apply(demodify_peptide)
merged_df['DemodifiedAnnotationOther'] = merged_df['AnnotationOther'].dropna().apply(demodify_peptide)

# Extract the demodified annotations for processing
demodified_annotations = merged_df['DemodifiedAnnotation']
demodified_annotations_other = merged_df['DemodifiedAnnotationOther']

# Create sets of unique demodified peptides
peptides_DB = set(demodified_annotations)
peptides_PA = set(demodified_annotations_other)
all_peptides = peptides_DB.union(peptides_PA)

# Iterate through all unique peptides
# Iterate through all unique peptides with progress bar
for peptide in tqdm(all_peptides, desc="Processing peptides"):
    count_DB = 0
    scans_DB = []
    count_PA = 0
    scans_PA = []
    
    # Precompute demodified annotations for the current peptide
    # Iterate through the DataFrame to count matches and collect scans
    matches_DB = merged_df[merged_df['DemodifiedAnnotation'] == peptide]
    count_DB = len(matches_DB)
    scans_DB = matches_DB['Scan'].tolist()
    
    matches_PA = merged_df[merged_df['DemodifiedAnnotationOther'] == peptide]
    count_PA = len(matches_PA)
    scans_PA = matches_PA['Scan'].tolist()
    
    # Append the results for this peptide
    results.append({
        'Peptide': peptide,
        'Count_DB': count_DB,
        'Scans_DB': scans_DB,
        'Count_PA': count_PA,
        'Scans_PA': scans_PA
    })


Processing peptides: 100%|██████████| 6109/6109 [00:27<00:00, 218.81it/s]


In [9]:
import csv

results
# Write the results to a TSV file
with open('peptide_level.tsv', 'w', newline='') as tsvfile:
    fieldnames = ['Peptide', 'Count_DB', 'Scans_DB', 'Count_PA', 'Scans_PA']
    writer = csv.DictWriter(tsvfile, fieldnames=fieldnames, delimiter='\t')
    
    # Write the header
    writer.writeheader()
    
    # Write the data
    writer.writerows(results)